# CICIDS2017 — Data Pipeline

This notebook orchestrates the full **data preparation** stage of the project:

| Step | Module | What it does |
|------|--------|--------------|
| 1 | `src.data_ingestion` | Load & merge raw CICIDS2017 CSVs from `Input/` |
| 2 | `src.preprocessing` | Clean data (dedup, inf values, missing values, label standardisation) |
| 3 | `src.feature_engineering` | Remove collinear & low-importance features |
| 4 | `src.eda` | *(optional)* Statistical analysis & visualisations |
| 5 | Output | Save `cicids2017_cleaned.csv` for model training |

> **Note on resampling & scaling:** SMOTE, RobustScaler, and similar transforms
> must be applied *inside* the training pipeline (after the train/test split) to
> avoid data leakage.  They are **not** applied here — see `02_model_training.ipynb`.

## 0. Setup

In [1]:
import sys
import os

# Ensure the project root is on the path so `src` is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_ingestion import load_raw_data
from src.preprocessing import run_preprocessing_pipeline
from src.feature_engineering import (
    get_feature_types,
    run_feature_engineering_pipeline,
)

print('All imports OK.')

ModuleNotFoundError: No module named 'pandas'

## 1. Data Ingestion

Load all eight CICIDS2017 CSV files from `Input/` and concatenate them into a single DataFrame.

In [ ]:
INPUT_DIR = os.path.join(PROJECT_ROOT, 'Input')

df_raw = load_raw_data(INPUT_DIR)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

## 2. Preprocessing

Run the full cleaning pipeline:
- Strip column names
- Remove duplicates
- Remove identity (all-same-values) columns
- Replace ±inf with NaN and drop NaN rows
- Rename `Label` → `Attack Type` and group attack subtypes

In [ ]:
df_clean = run_preprocessing_pipeline(df_raw)

# Free raw DataFrame from memory
del df_raw

print(f'Clean shape: {df_clean.shape}')
df_clean['Attack Type'].value_counts()

## 3. Feature Engineering

- Analyse feature correlations (heatmap)
- Drop highly collinear columns (|r| ≥ 0.95)
- Drop low-importance columns (from combined RF + Kruskal-Wallis analysis)

> Set `plot_heatmap=False` for faster, headless runs.

In [ ]:
df_final = run_feature_engineering_pipeline(
    df_clean,
    corr_threshold=0.85,
    plot_heatmap=True,
)

print(f'Final shape: {df_final.shape}')
df_final.head(3)

## 4. (Optional) EDA — Statistical Analysis

These cells run the Levene and Kruskal-Wallis tests plus an RF-based importance
check.  They are **informational** — the results were already used to determine
which columns to drop above.

Uncomment and run only when you want to reproduce or explore the analysis.

In [ ]:
# from src.eda import (
#     analyze_variance_homogeneity,
#     analyze_feature_importance_kruskal,
#     analyze_feature_importance_rf,
#     plot_feature_importance_combined,
# )
#
# num_feat, _ = get_feature_types(df_final)
#
# # Levene test
# variance_result = analyze_variance_homogeneity(df_final, num_feat)
#
# # Kruskal-Wallis
# h_df = analyze_feature_importance_kruskal(df_final, num_feat)
#
# # Random Forest importance
# importance_df, cm, rf_labels, cv_scores = analyze_feature_importance_rf(df_final, num_feat)
#
# # Combined visualisation
# comp = importance_df.merge(h_df, on='Feature', how='left').sort_values('Importance', ascending=False)
# plot_feature_importance_combined(comp)

## 5. Save Output

In [ ]:
OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'cicids2017_cleaned.csv')
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'Saved cleaned dataset → {OUTPUT_PATH}')
print(f'Shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns')

---

**Next step:** Open `02_model_training.ipynb` to perform the stratified train/test
split and train classification models using `src.model_training`.